# 01 — Data Preparation and Statistical Screening

Exploratory analysis of the coffee-market universe and the statistical
prerequisites for pairs trading:

1. load and clean the close-price dataset;
2. ADF unit-root tests — candidates must be non-stationary, i.e. I(1);
3. Engle-Granger cointegration tests against the Arabica anchor `KC1`;
4. selection of the 4-pair equity basket.

All screening is performed on the **first train window** of the walk-forward
validation (the first `min_train_years` of history), so the universe
selection never sees data that any fold later trades on.

> **Data note:** this notebook points at the Bloomberg export
> (`configs/bloomberg.yaml`, git-ignored). Without Bloomberg access, run
> `python scripts/download_data.py` and switch to `configs/default.yaml` —
> see `data/README.md` for the differences between the two sources.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

from src import backtest, data, hedging, metrics, signals, stat_tests, walk_forward

cfg = yaml.safe_load(open("../configs/bloomberg.yaml", encoding="utf-8")) # select default.yaml if Bloomberg data is not available
ANCHOR = cfg["universe"]["anchor"]
EQUITIES = cfg["universe"]["equities"]
WINDOW = cfg["signals"]["zscore_window"]
WF = cfg["walk_forward"]
RF = cfg["strategy"]["risk_free_rate"]
TC = cfg["strategy"]["transaction_cost"]

## 1. Load prices and inspect data quality

In [ ]:
prices = data.load_prices("../" + cfg["data"]["path"])
display(prices.tail())
display(data.data_quality_report(prices).head(10))

NaNs are concentrated in late-listed names (`BROS`, `WEST`, `THCH`, ...).
Rather than dropping rows globally — which would shrink the sample for every
asset — those equities are excluded from the candidate universe via the
config.

## 2. Arabica vs. Robusta futures prices

In [ ]:
## Quick visualization Arabica vs. Robusta Futures prices
# (DF1 is only available in the Bloomberg dataset)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
ax1.plot(prices.index, prices["KC1"], color="saddlebrown", linewidth=1.2, label="KC1")
ax2.plot(prices.index, prices["DF1"], color="green", linewidth=1.2, label="DF1")
ax1.set_ylabel("KC1 Price (US cents/lb)", color="saddlebrown")
ax2.set_ylabel("DF1 Price (USD/tonne)", color="green")
ax1.tick_params(axis="y", labelcolor="saddlebrown")
ax2.tick_params(axis="y", labelcolor="green")
ax1.set_xlim([pd.to_datetime('2016-02-12'), pd.to_datetime('2026-02-12')])
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left"); ax1.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Log-prices and the walk-forward fold structure

The shaded boxes show how the backtest is run: fold *k* calibrates on every
observation **before** its test box (expanding train window, starting from the
blue region) and then trades the test box out-of-sample. The thin white gaps
between boxes are the 1-month embargo. Only the shaded test folds contribute
to reported performance.

In [ ]:
log_prices = data.to_log_prices(prices)
log_screen = data.screening_window(log_prices, WF["min_train_years"])
print(f"Screening window: {log_screen.index[0].date()} -> "
      f"{log_screen.index[-1].date()} ({len(log_screen)} obs)")

# Walk-forward fold structure: the blue box is the first train window; each
# numbered box is a 12-month test fold, separated by the 1-month embargo.
# Train windows are EXPANDING: fold k trains on everything before its test box.
folds = walk_forward.walk_forward_folds(
    log_prices.index,
    min_train_size=int(WF["min_train_years"] * 252),
    test_size=WF["test_months"] * 21,
    embargo=WF["embargo_days"],
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(log_prices[ANCHOR], color="saddlebrown", lw=1.2)
ax.axvspan(log_prices.index[0], folds[0][0][-1],
           color="steelblue", alpha=0.15, label="first train window (expanding)")
for i, (train_idx, test_idx) in enumerate(folds):
    ax.axvspan(test_idx[0], test_idx[-1], color=["orange", "green"][i % 2],
               alpha=0.12, label="test folds" if i == 0 else None)
    mid = test_idx[len(test_idx) // 2]
    ax.text(mid, log_prices[ANCHOR].max(), f"{i+1}", ha="center",
            va="top", fontsize=9, fontweight="bold", color="dimgrey")
ax.set_ylabel("log price"); ax.legend(loc="upper left"); ax.grid(alpha=0.3)
ax.set_xlim([pd.to_datetime('2016-02-12'), pd.to_datetime('2026-02-12')])
plt.tight_layout()
plt.show()

## 4. ADF unit-root screening

In [ ]:
exclude = set(cfg["universe"]["exclude"]) | {ANCHOR}
adf_results = stat_tests.adf_screen(log_screen, exclude=exclude)
adf_results.round(4)

High p-values fail to reject the unit root: the series are I(1), as required.
A *stationary* price series would already be mean-reverting on its own and
admit no long-run equilibrium relationship to trade.

## 5. Engle-Granger cointegration against KC1

In [ ]:
eg_results = stat_tests.engle_granger_screen(log_screen, ANCHOR, exclude=exclude)
display(eg_results.round(4))

summary = stat_tests.summary_table(log_screen, ANCHOR, adf_results, eg_results)
summary.round(4)

The evidence of cointegration within the assigned equity universe is weak —
this is the **structural limitation of the exercise**, imposed by the
universe rather than the methodology: large consumer-staples companies pass
coffee input costs through to consumers, severing the long-run link between
their equity prices and the commodity. A more natural cointegrated pair
would have been Arabica vs. Robusta (`KC1`/`DF1`), two substitutable coffee
qualities linked by blending economics. The strategy instead relies on the
**multivariate basket**, letting the optimizer search for a synthetic
combination that mean-reverts better than any individual pair.

## 6. Visual check of the selected basket

In [ ]:
top4 = eg_results.head(4)["asset"].tolist()
fig, axes = plt.subplots(len(top4), 1, figsize=(12, 3.5 * len(top4)))
for ax, asset in zip(np.atleast_1d(axes), top4):
    pair = pd.concat([prices[ANCHOR], prices[asset]], axis=1).dropna()
    ax2 = ax.twinx()
    ax.plot(pair.index, pair[ANCHOR], color="tab:blue", lw=1.1, label=ANCHOR)
    ax2.plot(pair.index, pair[asset], color="tab:red", lw=1.1, label=asset)
    pval = eg_results.loc[eg_results["asset"] == asset, "p_value"].iloc[0]
    ax.set_title(f"{asset} vs {ANCHOR} (Engle-Granger $p$ = {pval:.4f})")
    ax.grid(alpha=0.3)
    ax.set_xlim([pd.to_datetime('2016-02-12'), pd.to_datetime('2026-02-12')])
plt.tight_layout(); plt.show()